In [ ]:
!pip uninstall -y unsloth unsloth-zoo
!pip cache purge
!pip install -U --no-cache-dir "unsloth[colab-new]" unsloth-zoo

Files removed: 0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 130.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 415.2/415.2 kB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 165.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 234.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 261.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 112.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 114.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 165.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 183.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.9/224.9 kB 275.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 269.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 MB 109.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━

In [ ]:
import unsloth
from unsloth import FastLanguageModel

import torch
import pandas as pd
import json
from huggingface_hub import HfApi, login
from google.colab import files
import io
from transformers import pipeline

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [ ]:
from google.colab import userdata
HF_TOKEN = userdata.get('Huggingface')

In [ ]:
login(token=HF_TOKEN)

Downloading Model and tokenizer

In [ ]:
model_name = 'unsloth/gemma-3-4B-it'
max_seq_length = 2048  # Choose any! We auto support RoPE Scaling internally!
dtype = None           # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+


In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = False
    # token = "hf_...", # use one if using gated models like meta-llama/Llama-2-7b-hf
)

==((====))==  Unsloth 2026.4.2: Fast Gemma3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma3 won't work! Using float32.
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/210 [00:00<?, ?B/s]

...

preprocessor_config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/670 [00:00<?, ?B/s]

In [ ]:
from unsloth.chat_templates import get_chat_template

# chat template 적용 (pipeline 사용 전 한 번만 실행)
tokenizer = get_chat_template(
    tokenizer,
    chat_template='gemma-3'
)

text_generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=128,
    return_full_text=False,  # 입력 프롬프트 제외하고 생성분만 반환
)

def get_response(prompt: str) -> str:
    # chat template 형식으로 변환 후 전달
    messages = [{"role": "user", "content": prompt}]
    formatted = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    sequences = text_generator(formatted)
    return sequences[0]['generated_text']

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [ ]:
prompt = 'What is Machine Learning?'
response = get_response(prompt)
response

Passing `generation_config` together with generation-related arguments=({'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


"Okay, let's break down what Machine Learning is all about. It's a really fascinating and rapidly growing field, and it can seem a bit complex at first. Here's a breakdown in layers:\n\n**1. The Basic Idea: Teaching Computers to Learn**\n\nAt its core, Machine Learning (ML) is about enabling computers to learn from data *without* being explicitly programmed for every single task.  Traditionally, to make a computer do something, you'd have to write specific instructions for *everything*.  With ML, you give the computer a lot of data and let it figure out the patterns and rules"

In [ ]:
prompt = 'What is CallCrazy service?'
response = get_response(prompt)
response

Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


'CallCrazy is a unique and somewhat quirky service that essentially lets you **"rent" a phone number to receive calls and texts for a specific purpose, and then have someone else monitor and respond on your behalf.** It\'s designed to handle situations where you don\'t want to be directly involved in a conversation, or where you need a professional-sounding contact.\n\nHere\'s a breakdown of what it offers and how it works:\n\n**Key Features & What You Can Do With It:**\n\n* **Rental Numbers:** You can rent a temporary phone number (or multiple) for a variety of uses.\n* **Monitoring &'

In [ ]:
from datasets import load_dataset

def convert_to_chat_format(examples):
    """ 기존 데이터셋을 'messages' 형식으로 변환 """
    messages_list = []
    for question, response in zip(examples["input"], examples["response"]):
        messages = [
            {"role": "user", "content": question.strip()},
            {"role": "assistant", "content": response.strip()},
        ]
        messages_list.append(messages)
    return {"messages": messages_list}

dataset = load_dataset("JaeminKim/Callcrazy", split="train")
dataset = dataset.map(convert_to_chat_format, batched=True)

call_crazy.jsonl: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/10 [00:00<?, ? examples/s]

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

In [ ]:
def apply_chat_template(examples) :
  texts = tokenizer.apply_chat_template(examples['messages'])
  return { "text" : texts }

dataset = dataset.map(apply_chat_template, batched = True)
dataset

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

Dataset({
    features: ['instruction', 'input', 'response', 'messages', 'text'],
    num_rows: 10
})

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth: Making `base_model.model.model.vision_tower.vision_model.embeddings` require gradients


In [ ]:
from trl import SFTTrainer, SFTConfig
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    eval_dataset = None,
    args = SFTConfig(
        dataset_text_field = 'text',
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        # num_train_epochs = 1, # Set this for 1 full training run.
        max_steps = 100,
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none",
    ),
)

Unsloth: Switching to float32 training since model cannot work with float16


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/10 [00:00<?, ? examples/s]

In [ ]:
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(
    trainer,
    instruction_part = '<start_of_turn>user\n',
    response_part = '<start_of_turn>model\n',
)

Map (num_proc=6):   0%|          | 0/10 [00:00<?, ? examples/s]

Filter (num_proc=6):   0%|          | 0/10 [00:00<?, ? examples/s]

In [ ]:
import wandb
wandb.init(mode="disabled")
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 10 | Num Epochs = 50 | Total steps = 100
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 32,788,480 of 4,332,867,952 (0.76% trained)


Step,Training Loss
1,7.086338
2,7.239282
3,6.460108
4,5.526670
5,4.179122
6,3.232635
7,2.753305
8,1.959826
9,1.899601
10,1.512433


In [ ]:
def get_model_response(question):
    # 메시지 형식으로 변환 (Gemma-3 채팅 템플릿 사용)
    messages = [{
        'role': 'user',
        'content': [{
            'type': 'text',
            'text': question,
        }]
    }]

    # 채팅 템플릿 적용
    text = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True
    )

    # 모델 입력을 위한 토큰화
    inputs = tokenizer(
        [text],
        return_tensors="pt"
    ).to("cuda")

    # 모델 응답 생성
    outputs = model.generate(
        **inputs,
        max_new_tokens=1024,
        temperature=1.0,
        top_p=0.95,
        top_k=64,
    )

    # 모델 응답 디코딩
    generated_text = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]

    return generated_text


In [ ]:
question = 'How much is Callcrazy service?'
print(get_model_response(question))

In [ ]:
question = 'What is the uniqueness of Callcrazy service ?'
print(get_model_response(question))

In [ ]:
question = 'Explain about Seoul'
print(get_model_response(question))


user
How much is Callcrazy service?
model
The basic fee is 5,000 won, and 1,000 won is added per km
user
What is the uniqueness of Callcrazy service ?
model
We show the current location of the shipment on a map and deliver the product inexpensively and reliably.
user
Explain about Seoul
model
Seoul is the capital and most populous city of South Korea. It's a major financial, industrial, and cultural center, known for its fast-paced lifestyle, modern architecture, traditional palaces, and shopping.


In [ ]:
from huggingface_hub import login, HfApi
from google.colab import userdata

HF_TOKEN = userdata.get('Huggingface')
login(token=HF_TOKEN)

save_path = "fine-tuned-gemma-model"

# LoRA 적용된 모델 저장
trainer.model.save_pretrained(save_path)

# tokenizer 저장
trainer.processing_class.save_pretrained(save_path)

# Hugging Face Hub 업로드
repo_name = "JaeminKim/Gemma_callme"
api = HfApi()
api.create_repo(repo_name, exist_ok=True)

trainer.model.push_to_hub(repo_name)  # Model 업로드
trainer.processing_class.push_to_hub(repo_name)   # tokenizer 업로드

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          | 60.6kB /  131MB            

Saved model to https://huggingface.co/JaeminKim/Gemma_callme


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mprc2_7f82/tokenizer.json:  47%|####6     | 15.7MB / 33.4MB            

No files have been modified since last commit. Skipping to prevent empty commit.


In [ ]:
!pip uninstall -y unsloth unsloth-zoo
!pip cache purge
!pip install -U --no-cache-dir "unsloth[colab-new]" unsloth-zoo

Files removed: 0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 193.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 415.2/415.2 kB 72.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 182.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 253.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 236.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 136.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 232.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 143.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 232.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.9/224.9 kB 249.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 253.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 MB 167.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━

In [ ]:
import torch
from unsloth import FastLanguageModel

# Unsloth가 Hub repo에서 베이스 모델 + LoRA 어댑터를 자동으로 결합
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="JaeminKim/Gemma_callme",  # Fine-tuned repo
    max_seq_length=4096,
    dtype=None,          # None → bf16/fp16 자동 감지
    load_in_4bit=True,   # 학습 시와 동일 설정 유지
)

# 추론 모드 활성화 (Unsloth 내부 최적화 커널 적용)
FastLanguageModel.for_inference(model)

# device 확인만 (이동은 하지 않음)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"모델 디바이스: {device}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.2: Fast Gemma3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma3 won't work! Using float32.


model.safetensors:   0%|          | 0.00/4.56G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/210 [00:00<?, ?B/s]

processor_config.json:   0%|          | 0.00/70.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/670 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/131M [00:00<?, ?B/s]

모델 디바이스: cuda


In [ ]:
def get_model_response(question):
    # 메시지 형식으로 변환 (Gemma-3 채팅 템플릿 사용)
    messages = [{
        'role': 'user',
        'content': [{
            'type': 'text',
            'text': question,
        }]
    }]

    # 채팅 템플릿 적용
    text = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True
    )

    # 모델 입력을 위한 토큰화
    inputs = tokenizer(
        [text],
        return_tensors="pt"
    ).to("cuda")

    # 모델 응답 생성
    outputs = model.generate(
        **inputs,
        max_new_tokens=1024,
        temperature=1.0,
        top_p=0.95,
        top_k=64,
    )

    # 모델 응답 디코딩
    generated_text = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]

    return generated_text


In [ ]:
question = 'How much is Callcrazy service?'
print(get_model_response(question))


user
How much is Callcrazy service?
model
The basic fee is 5,000 won, and 1,000 won is added per km
user
What is the uniqueness of Callcrazy service ?
model
We show the current location of the shipment on a map and deliver the product inexpensively and reliably.
user
Explain about Seoul
model
Seoul is the capital and largest city of South Korea, located in the northwestern part of the country. It is a major economic, financial, and cultural center, and is known for its fast pace, modern architecture, traditional palaces, and shopping.


In [ ]:
question = 'What is the uniqueness of Callcrazy service ?'
print(get_model_response(question))

In [ ]:
question = 'Explain about Seoul'
print(get_model_response(question))